## Data Ingestion (Streaming)

## Execution Notes and Batch Logging
This notebook continuously aggregates Kafka messages into Landing Zone JSON files. Every polling cycle is logged as a streaming batch with start time, new records, previous totals, updated totals, output keys, and committed offsets.


Here, we store the streaming data coming from **Kafka** as aggregated data in our Landing Zone.

**Landing governance mirror note.** Kafka aggregation objects written by this notebook carry the same lightweight governance metadata fields as the ingestion DAG, but the notebook defines them locally and does not import DAG helpers.


**Importing Useful Libraries**

In [1]:
from kafka import KafkaConsumer
from dotenv import load_dotenv
import json
import boto3
import io
import os
import time
from datetime import datetime, timezone

# Load .env file
load_dotenv()

# Read environment variables
endpoint = os.getenv("MINIO_ENDPOINT")
MINIO_ROLE = "writer"
if MINIO_ROLE == "admin":
    access_key = os.getenv("MINIO_ACCESS_KEY")
    secret_key = os.getenv("MINIO_SECRET_KEY")
else:
    role_prefix = MINIO_ROLE.upper()
    access_key = os.getenv(f"MINIO_{role_prefix}_ACCESS_KEY")
    secret_key = os.getenv(f"MINIO_{role_prefix}_SECRET_KEY")
if not endpoint or not access_key or not secret_key:
    raise RuntimeError(f"Missing MinIO {MINIO_ROLE} credentials in environment")

# Notebook-local governance metadata mirror for Landing Zone Kafka aggregation outputs
LANDING_GOVERNANCE_DEFAULTS = {
    "owner": "data_engineering_team",
    "data_steward": "bdm_project_team",
    "data_classification": "public_environmental_observation",
    "pii_flag": "no_direct_pii",
    "retention_policy": "course_project_retained_until_assessment_archive",
}

def landing_object_metadata(topic_name, object_key):
    now = datetime.now(timezone.utc).isoformat()
    return {
        "source": "kafka",
        "topic": topic_name,
        "source_system": "kafka",
        "ingestion_time": now,
        "source_file_path": f"kafka://{topic_name}/notebook-aggregation",
        "logical_date": now,
        "validation_status": "valid",
        "schema_version": "landing_raw_v1",
        "landing_name": object_key,
        **LANDING_GOVERNANCE_DEFAULTS,
    }

# Deserializer function
def deserialize(m):
    return json.loads(m.decode("utf-8"))

In [2]:
# -------------------------
# Kafka Consumers
# -------------------------
consumer_weather = KafkaConsumer(
    'weather-barcelona',
    group_id='weather-group',
    bootstrap_servers='kafka:9092',
    value_deserializer=deserialize,
    auto_offset_reset='earliest',
    enable_auto_commit=False,
)

consumer_air = KafkaConsumer(
    'airquality-barcelona',
    group_id='airquality-group',
    bootstrap_servers='kafka:9092',
    value_deserializer=deserialize,
    auto_offset_reset='earliest',
    enable_auto_commit=False,
)

In [3]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint, # MinIO API endpoint
    aws_access_key_id=access_key, # User name
    aws_secret_access_key=secret_key, # Password
)

In [ ]:
# -------------------------
# Aggregation loop
# -------------------------
bucket_name = 'landing-zone'

# Fetch new messages given a consumer
def fetch_new_messages(consumer):
    records = consumer.poll(timeout_ms=3000)

    rows = []
    for tp, msgs in records.items():
        for msg in msgs:
            rows.append(msg.value)

    return rows

# Load an existing file
def load_existing(bucket_name, key):
    try:
        obj = s3.get_object(Bucket=bucket_name, Key=key)
        return json.loads(obj['Body'].read())
    except s3.exceptions.NoSuchKey:
        return []

# Aggregated file paths
weather_agg_key = 'persistent-landing/semistructured/weather-barcelona.json'
air_agg_key = 'persistent-landing/semistructured/airquality-barcelona.json'

batch_id = 0
while True:

    batch_id += 1
    batch_started_at = datetime.now(timezone.utc).isoformat()
    print(f"\n[stream batch {batch_id}] started at {batch_started_at}")

    # ----- Weather -----
    weather_new = fetch_new_messages(consumer_weather)
    weather_existing = load_existing(bucket_name, weather_agg_key)

    weather_updated = weather_existing + weather_new

    s3.put_object(
        Bucket=bucket_name,
        Key=weather_agg_key,
        Body=json.dumps(weather_updated),
        ContentType="application/json",
        Metadata=landing_object_metadata("weather-barcelona", weather_agg_key)
    )

    print(f"[stream batch {batch_id}] weather: new={len(weather_new)}, previous={len(weather_existing)}, total={len(weather_updated)}, key={weather_agg_key}")

    # ----- Air Quality -----
    air_new = fetch_new_messages(consumer_air)
    air_existing = load_existing(bucket_name, air_agg_key)

    air_updated = air_existing + air_new

    s3.put_object(
        Bucket=bucket_name,
        Key=air_agg_key,
        Body=json.dumps(air_updated),
        ContentType="application/json",
        Metadata=landing_object_metadata("airquality-barcelona", air_agg_key)
    )

    print(f"[stream batch {batch_id}] air quality: new={len(air_new)}, previous={len(air_existing)}, total={len(air_updated)}, key={air_agg_key}")

    # Commit offsets AFTER successful processing
    consumer_weather.commit()
    consumer_air.commit()
    print(f"[stream batch {batch_id}] offsets committed; sleeping 60 seconds")

    # Wait before next batch
    time.sleep(60)


[stream batch 1] started at 2026-06-06T16:45:26.550041+00:00
[stream batch 1] weather: new=13, previous=0, total=13, key=persistent-landing/semistructured/weather-barcelona.json
[stream batch 1] air quality: new=13, previous=0, total=13, key=persistent-landing/semistructured/airquality-barcelona.json
[stream batch 1] offsets committed; sleeping 60 seconds

[stream batch 2] started at 2026-06-06T16:46:27.732338+00:00
[stream batch 2] weather: new=1, previous=13, total=14, key=persistent-landing/semistructured/weather-barcelona.json
[stream batch 2] air quality: new=1, previous=13, total=14, key=persistent-landing/semistructured/airquality-barcelona.json
[stream batch 2] offsets committed; sleeping 60 seconds

[stream batch 3] started at 2026-06-06T16:47:27.814825+00:00
[stream batch 3] weather: new=1, previous=14, total=15, key=persistent-landing/semistructured/weather-barcelona.json
[stream batch 3] air quality: new=1, previous=14, total=15, key=persistent-landing/semistructured/airqu

KeyboardInterrupt: 